In [0]:
#!pip install lseg-data
#%pip install lseg-data --quiet

In [0]:
# Run these commands in a separate terminal or notebook cell to set up secrets:
# This cell is for reference only - run these commands once to store credentials securely

# Create secret scope (run once)
# databricks secrets create-scope refinitiv_scope

# Add secrets (run once for each secret)
# databricks secrets put-secret refinitiv_scope app_key --string-value "ea8c069e8cef427c84d4a777fe002f1cdbe929a0"
# databricks secrets put-secret refinitiv_scope username --string-value "nvaldez@tec.mx"
# databricks secrets put-secret refinitiv_scope password --string-value "cibgiz-1bipwy-baqjyK"

print("\nAfter running the above commands, use the cell below to retrieve secrets securely.")
print("Note: These commands should be run in Databricks CLI or using the Secrets API.")

In [0]:
# Create secret scope and store credentials securely
# Run this cell ONCE to set up your secrets

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Create secret scope
try:
    w.secrets.create_scope(scope="refinitiv_scope")
    print("✓ Secret scope 'refinitiv_scope' created successfully")
except Exception as e:
    print(f"Note: {e} (scope may already exist)")

# Store credentials
try:
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="app_key",
        string_value="e0"
    )
    print("✓ app_key stored")
    
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="username",
        string_value="nvaldez@tec.mx"
    )
    print("✓ username stored")
    
    w.secrets.put_secret(
        scope="refinitiv_scope",
        key="password",
        string_value="cKccccccccccccccccc"
    )
    print("✓ password stored")
    
    print("\n✓ All credentials stored securely in Databricks Secrets!")
except Exception as e:
    print(f"Error storing secrets: {e}")

✓ Secret scope 'refinitiv_scope' created successfully
✓ app_key stored
✓ username stored
✓ password stored

✓ All credentials stored securely in Databricks Secrets!


In [0]:
# Fetch Apple stock prices for the last year using ld API
import lseg.data as ld
from datetime import datetime, timedelta

# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")


Session opened successfully: OpenState.Opened
Session set as default


In [0]:
import lseg.data as ld

# Check version
print("lseg.data version:", ld.__version__)

# Check if we need to set the session as default
print("\nChecking session management methods:")
if hasattr(ld, 'set_default_session'):
    print("  ✓ ld.set_default_session() exists")
if hasattr(ld.session, 'set_default'):
    print("  ✓ ld.session.set_default() exists")

# List session-related methods
print("\nSession-related methods in ld:")
for attr in sorted(dir(ld)):
    if 'session' in attr.lower() and not attr.startswith('_'):
        print(f"  - {attr}")

lseg.data version: 2.1.1

Checking session management methods:
  ✓ ld.session.set_default() exists

Session-related methods in ld:
  - close_session
  - open_session
  - session


In [0]:

# Calculate date range for last year
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

# Fetch historical data using ld.get_history (not get_data)
df = ld.get_history(

    universe="AAPL.O",
    fields=["TR.PriceClose.date", "TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.Volume"],
    interval="1D",
    start=start_date,
    end=end_date
)

# Display the result
display(df)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ec158878-33fc-4dbb-9738-9fef7f6046a9/lib/python3.12/site-packages/lseg/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


Date,Price Close,Price Open,Price High,Price Low,Volume
2025-08-26T00:00:00.000Z,229.31,226.87,229.49,224.69,54575107
2025-08-27T00:00:00.000Z,230.49,228.61,230.9,228.26,31259513
2025-08-28T00:00:00.000Z,232.56,230.82,233.41,229.335,38074700
2025-08-29T00:00:00.000Z,232.14,232.51,233.38,231.37,39418437
2025-09-02T00:00:00.000Z,229.72,229.25,230.85,226.97,44075638
2025-09-03T00:00:00.000Z,238.47,237.21,238.85,234.36,66427835
2025-09-04T00:00:00.000Z,239.78,238.45,239.8999,236.74,47549429
2025-09-05T00:00:00.000Z,239.69,239.995,241.32,238.4901,54870397
2025-09-08T00:00:00.000Z,237.88,239.3,240.15,236.34,48999495
2025-09-09T00:00:00.000Z,234.35,237.0,238.7805,233.36,66313918
